In [11]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import numpy as np

# ==========================================
# 1. DEFINE THE NETWORK
# ==========================================
class surrogateDNN(nn.Module):
    # Removed 'hidden_size' argument since we hardcoded the sizes for options pricing
    def __init__(self, input_size, output_size):
        super(surrogateDNN, self).__init__() # CRITICAL FIX: Initializes PyTorch backend
        
        self.fc1 = nn.Linear(input_size, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 64)
        self.fc4 = nn.Linear(64, output_size)
        
        # LeakyReLU keeps small gradients alive better than ReLU
        self.activation = nn.LeakyReLU(0.01)
        self.dropout = nn.Dropout(0.1)

    def forward(self, x):
        x = self.activation(self.fc1(x))
        x = self.activation(self.fc2(x))
        x = self.activation(self.fc3(x))
        x = self.fc4(x) 
        return x


def prepare_data(df, target_col, batch_size=256): # Increased batch_size for speed
    """Handles splitting, scaling (features AND targets), and DataLoaders."""
    X = df.drop(target_col, axis=1)
    y = df[[target_col]] # Keep as 2D DataFrame for the scaler

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Scale Features
    x_scaler = StandardScaler()
    X_train_scaled = x_scaler.fit_transform(X_train)
    X_test_scaled = x_scaler.transform(X_test) 

    # Scale Targets
    y_scaler = MinMaxScaler()
    y_train_scaled = y_scaler.fit_transform(y_train)
    y_test_scaled = y_scaler.transform(y_test)

    # Convert to Tensors
    X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
    X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
    y_train_tensor = torch.tensor(y_train_scaled, dtype=torch.float32) 
    y_test_tensor = torch.tensor(y_test_scaled, dtype=torch.float32)

    # Create Loaders
    train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
    test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

    train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False)

    # Return the y_scaler so we can "un-scale" the predictions later!
    return train_loader, test_loader, X.shape[1], y_scaler



if __name__ == "__main__":
    df = pd.read_csv("synthetic_labeled_options.csv")

    target_column_name = df.columns[-1]
    
    train_loader, test_loader, input_features, y_scaler = prepare_data(df, target_col=target_column_name)

    model = surrogateDNN(input_size=input_features, output_size=1)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    # Added a learning rate scheduler to help hit that 1-cent precision
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5)

    # Bumped to 100 epochs since option pricing is complex
    EPOCHS = 100
    print(f"Beginning training for {EPOCHS} epochs...")
    
    for epoch in range(EPOCHS):
        model.train()
        epoch_loss = 0
        
        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()
            predictions = model(batch_X)
            loss = criterion(predictions, batch_y)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
            
        avg_train_loss = epoch_loss / len(train_loader)
        scheduler.step(avg_train_loss) # Tells the scheduler to check if we should slow down learning
        
        if (epoch + 1) % 10 == 0:
            print(f'Epoch [{epoch+1}/{EPOCHS}], Scaled Loss: {avg_train_loss:.6f}')

    # --- E. Evaluation ---
    print("\nEvaluating model on test data...")
    model.eval()

    all_preds = []
    all_targets = []

    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            # Removed the extra .unsqueeze(1) that was causing shape mismatch
            outputs = model(batch_X)
            all_preds.append(outputs)
            all_targets.append(batch_y)

    # Combine all batches
    all_preds = torch.cat(all_preds, dim=0).numpy()
    all_targets = torch.cat(all_targets, dim=0).numpy()

    # CRITICAL: Inverse Transform to get real dollars back instead of 0-1 scale
    real_preds = y_scaler.inverse_transform(all_preds)
    real_targets = y_scaler.inverse_transform(all_targets)

    # Calculate Real-World Metrics
    mae = np.mean(np.abs(real_preds - real_targets))
    mse = np.mean((real_preds - real_targets)**2)
    rmse = np.sqrt(mse)
    
    # NEW: Max Squared Error
    max_sq_error = np.max((real_preds - real_targets)**2)

    # Calculate "Within 1 Cent" Accuracy
    within_penny = np.mean(np.abs(real_preds - real_targets) <= 0.01) * 100


    print(f'Mean Absolute Error (MAE): ${mae:.4f}')
    print(f'Root Mean Squared Error (RMSE): ${rmse:.4f}')
    print(f'Max Squared Error: ${max_sq_error:.4f}')
    print(f'Predictions within 1 cent: {within_penny:.2f}%')
    print("Pipeline complete!")

Beginning training for 100 epochs...


RuntimeError: mat1 and mat2 shapes cannot be multiplied (256x7 and 512x256)